In [2]:
from datetime import datetime, timedelta

def date_list(n_days):
    date_list = []
    today = datetime.now()

    for i in range(n_days):
        target_date = today - timedelta(days=i)
        date_list.append(target_date.strftime('%Y%m%d'))
    
    return date_list

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import pandas as pd
import time

def naver_sports_list_crawling(date_list):
    service = Service()
    options = Options()
    prefs = {
    "profile.managed_default_content_settings.images": 2,
    "safebrowsing.enabled": True
    }
    options.add_experimental_option('prefs', prefs)
    driver = webdriver.Chrome(service=service, options=options)

    article_links_dict = {}

    for date in date_list:
        list_url = f'https://m.sports.naver.com/wfootball/news?sectionId=wfootball&sort=popular&date={date}&isPhoto=N'
        try:
            driver.get(list_url)
            time.sleep(1)
            
            article_links = driver.find_elements(By.CSS_SELECTOR, 'li.NewsItem_news_item__fhEmd > a')

            url_list = []

            for link in article_links:
                full_url = f"{link.get_attribute('href')}"
                url_list.append(full_url)

            article_links_dict[date] = url_list

        except Exception as e:
            print(e)
            continue
    
    driver.quit()
    df = pd.DataFrame(
        [(date, url) for date, urls in article_links_dict.items() for url in urls],
        columns=['date', 'url']
    )
    df.to_csv('../data/raw/article_urls.csv', index=False)

    return df

In [51]:
print(save_urls(naver_sports_list_crawling(date_list(365))))

           date                                                url
0      20260304  https://m.sports.naver.com/wfootball/article/3...
1      20260304  https://m.sports.naver.com/wfootball/article/3...
2      20260304  https://m.sports.naver.com/wfootball/article/0...
3      20260304  https://m.sports.naver.com/wfootball/article/4...
4      20260304  https://m.sports.naver.com/wfootball/article/1...
...         ...                                                ...
14570  20250305  https://m.sports.naver.com/wfootball/article/1...
14571  20250305  https://m.sports.naver.com/wfootball/article/3...
14572  20250305  https://m.sports.naver.com/wfootball/article/1...
14573  20250305  https://m.sports.naver.com/wfootball/article/4...
14574  20250305  https://m.sports.naver.com/wfootball/article/1...

[14575 rows x 2 columns]


In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import pandas as pd
import time

def sports_article_crawling(date_list, df):
    service = Service()
    options = Options()
    prefs = {
    "profile.managed_default_content_settings.images": 2,
    "safebrowsing.enabled": True
    }
    options.add_experimental_option('prefs', prefs)

    for date in date_list:
        driver = webdriver.Chrome(service=service, options=options)
        titles = []
        contents = []
        dates = []

        urls = df[df['date']==int(date)]['url']
        for url in urls:
            try:
                driver.get(url)
                time.sleep(1)

                title_tag = driver.find_element(By.CSS_SELECTOR, 'h2.ArticleHead_article_title__qh8GV')
                content_tag = driver.find_element(By.CSS_SELECTOR, 'div._article_content')
               
                titles.append(title_tag.text)
                contents.append(content_tag.text)
                dates.append(date)

            except Exception as e:
                print(e)
                continue
        driver.quit()

        article_df = pd.DataFrame({
            "date": dates,
            "title": titles,
            "content": contents
        })
        article_df.to_csv(f'../data/raw/articles/{date}.csv', index=False, encoding='utf-8')

    return

In [6]:
df = pd.read_csv('../data/raw/article_urls.csv')
dates = date_list(365)
sports_article_crawling(dates, df)